# التقييم النهائي — MobileNet قبل تعزيز المتانة وبعد حقبتين

خلية مستقلة واحدة. فعّل GPU وأبقِ ملفات البيانات والتقسيم والأوزان وإيصالاتها في مساراتها. لا تُشغّل خلية التدريب بالتزامن. لا حاجة إلى متغيرات من الخلايا السابقة.

## ما سيحدث

1. التحقق من بصمة أوزان الحقبة2 وإيصالها، ومن أوزان المرحلة1 التي بدأت منها بالضبط. تُبحث النسخة في `robust_aug_phase2_v1/source_original.pt` أو في المسار الأصلي.
2. تحميل القائمة النظيفة المحفوظة والتحقق من توقيعها وحقولها مقابل CSV الأصلي. لا تُعاد حساب بصمات مليون صورة ولا يُنشأ تقسيم جديد. إذا لم توجد قائمة سليمة، يتوقف الفحص بوضوح.
3. تقييم النموذجين على Test الأصلي كاملًا:146425 صورة، دون تعزيز، FP32، عتبة0.5.
4. عينة ثابتة متوازنة من Test:500 لكل فئة، seed42. تجرى التحويلات السبعة المحددة مسبقًا للنموذجين على الصور نفسها. تُفحص قابلية فك QR ومطابقة المحتوى مرة لكل تحويل، دون فتح الروابط أو حفظ نصوصها.
5. حفظ المقاييس والتوقعات والتقارير والرسومات والبصمات داخل مجلد جديد، وتجميعها في ZIP. ملفات الأوزان ليست ضمن ZIP النتائج؛ احتفظ بحزمة الأوزان التي نزّلتها سابقًا.

## تفسير النتائج والمنهجية

- أوزان المرحلة2 مثبّتة عند `best_epoch_02_796c6ee005f6.pt`؛ لا اختيار حقب أو عتبات باستخدام Test.
- Test سبق فتحه لإصدار تقرير المرحلة1. هذا تقييم موثق بعد تصميم التعزيزات واختيار الحقبة على Validation؛ لا ندعي أن مجموعة الاختبار لم تستخدم من قبل. لا تتخذ تعديلات تدريب جديدة بناء على هذه النتائج دون توثيق دورة تطوير جديدة وحدودها.
- إزاحات4بكسلات لا تعيد التحجيم. لتجنب قص محتوى، تستبعد فقط الصور التي لا تملك4بكسلات بيضاء على كل الأطراف من حالات الإزاحة، ويُبلّغ عددها. الأصل والتصغير يقيمان كامل العينة. لا اختيار بناءً على صحة القرار. المقارنة بين النموذجين تستخدم الأفراد أنفسهم داخل كل حالة؛ فرق حالة الإزاحة عن الأصل يقارن أيضًا بالأصل على المجموعة المطابقة.
- التصغير224→208 يوضع في وسط لوحة224 بيضاء. Nearest وBilinear منفصلان.
- المتوسط على ست حالات محوّلة ملخص وصفي للصور المتكررة عبر التحويلات، وليس عينة6000صورة مستقلة. عينة المتانة متوازنة بخلاف Test الكامل، فلا تُقارن نسب الدقة بينهما دون هذا التوضيح.
- إخفاق OpenCV في فك QR لا يثبت عدم قابليته للقراءة؛ تبلَّغ نسبة المطابقة على الصور المقروءة أصلًا.
- جدول الاختبار الكامل يعرض Accuracy وMacro-F1 وPrecision/Recall/F1 للخبيث وAUC وAP. المقارنات وصفية؛ لم تنفذ اختبارات دلالة إحصائية أو تجارب متعددة البذور.

## التشغيل والوقت

قد يستغرق التقييم الكامل للنموذجين وفك QR عدة دقائق أو أكثر، وفق تخزين Kaggle وسرعة CPU. تُعرض مراحل وشريط تقدم. عاملان لتحميل Test وخيط CPUواحد؛ إذا واجهت خطأ عمال، يمكن تغيير WORKERSإلى0 وإعادة التقييم، دون تغيير الأوزان أو العتبة.

لا تثبت مكتبات جديدة أثناء الجلسة؛ PyTorch/torchvision/pandas/numpy/Pillow/scikit-learn/pyarrow/matplotlib/tqdm/OpenCV مستخدمة في الدفاتر السابقة. إذا انقطع التنفيذ، تُحفظ نتائج كل مرحلة مكتملة في مجلدها؛ إعادة الخلية تنشئ مجلد تقييم جديدًا.

أرسل `full_test_comparison.csv` و`test_robustness_comparison.csv` و`test_robustness_summary.csv` بعد النهاية.

فُحصت صياغة الكود ووظائف التحويل والمقاييس محليًا؛ لم ينفذ GPU أو يقرأ أوزانك من طرف المساعد.


In [ ]:
print('FINAL EVALUATION: phase-1 vs phase-2 epoch-2. No training.',flush=True)
import os,json,time,uuid,hashlib,gc,zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms as T
from PIL import Image,ImageChops
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,f1_score,precision_score,
    recall_score,roc_auc_score,average_precision_score,confusion_matrix,classification_report,
    ConfusionMatrixDisplay,roc_curve,precision_recall_curve)

ROOT=Path('/kaggle/working/qr_visual_fresh')
RUN=Path('/kaggle/working/mobilenet_continue/robust_aug_phase2_v1')
NEW_WEIGHTS=RUN/'best_epoch_02_796c6ee005f6.pt'
ORIGINAL_CSV=Path('/kaggle/input/datasets/feras2004/spliteoriginal/splitsOriginal.csv')
EXPECTED_SIGNATURE='d4509181f834a16144ad45e24b8ce686a4e608e07a9846a880757c705739d1ff'
THRESHOLD=0.5
BATCH_SIZE=128
WORKERS=2
SAMPLE_PER_CLASS=500
SAMPLE_SEED=42
VARIANTS=['original','translate_left_4','translate_right_4','translate_up_4',
          'translate_down_4','shrink_nearest_208','shrink_bilinear_208']
assert torch.cuda.is_available(),'Enable GPU first. CUDA must be True.'
DEVICE=torch.device('cuda:0')
torch.set_num_threads(1)
cv2.setNumThreads(1)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
print('GPU:',torch.cuda.get_device_name(0),flush=True)

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def write_json(value,path):
    path=Path(path);tmp=path.with_name(path.name+'.writing')
    with tmp.open('w',encoding='utf-8') as f:
        json.dump(value,f,ensure_ascii=False,indent=2,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def signature(frame):
    h=hashlib.sha256()
    cols=['path','label','split','url','ok','sha256']
    for row in tqdm(frame[cols].itertuples(index=False,name=None),total=len(frame),
                    desc='Verify data signature',mininterval=2):
        h.update((json.dumps(row,ensure_ascii=False,separators=(',',':'))+'\n').encode())
    return h.hexdigest()

print('[1/5] Verify both checkpoints...',flush=True)
assert NEW_WEIGHTS.is_file(),f'Missing: {NEW_WEIGHTS}'
new_receipt=json.loads(NEW_WEIGHTS.with_suffix('.json').read_text())
assert new_receipt['file']==NEW_WEIGHTS.name
NEW_SHA=file_hash(NEW_WEIGHTS)
assert NEW_SHA==new_receipt['sha256'],'Phase-2 checkpoint checksum mismatch.'
new_checkpoint=torch.load(NEW_WEIGHTS,map_location='cpu',weights_only=False) # your own trusted file
assert new_checkpoint['epoch']==2
assert new_checkpoint['data_signature']==EXPECTED_SIGNATURE
assert new_checkpoint['cfg']['max_train'] is None
assert new_checkpoint['cfg']['img_size']==224
OLD_SHA=new_checkpoint['source_sha256']
old_candidates=[RUN/'source_original.pt',Path('/kaggle/working/mobilenet_continue/full_train_phase1_fp32/best_epoch_04_10a6296afd7d.pt')]
OLD_WEIGHTS=None
for path in old_candidates:
    if path.is_file() and file_hash(path)==OLD_SHA:
        OLD_WEIGHTS=path;break
assert OLD_WEIGHTS is not None,'Restore the original phase-1 weights or source_original.pt with the matching saved SHA256.'
old_checkpoint=torch.load(OLD_WEIGHTS,map_location='cpu',weights_only=False)
assert old_checkpoint['epoch']==4 and old_checkpoint['data_signature']==EXPECTED_SIGNATURE
STATES={'phase1':old_checkpoint['model_state'],'phase2_epoch2':new_checkpoint['model_state']}
for name,state in STATES.items():
    for key,value in state.items():
        assert torch.is_tensor(value),key
        if value.is_floating_point():assert torch.isfinite(value).all(),f'{name}: nonfinite {key}'
print('PASS: phase-2 weights and exact phase-1 source verified.',flush=True)

print('[2/5] Verify saved Test membership...',flush=True)
audit=json.loads((ROOT/'audit.json').read_text())
assert audit['data_signature']==EXPECTED_SIGNATURE and audit.get('train_ready') is True
df=None;manifest_used=None
for path in [ROOT/'verified_mobilenet_inputs/splits_verified.parquet',
             ROOT/'verified_mobilenet_inputs/splits_verified.csv.gz',ROOT/'splits_clean.parquet']:
    if not path.is_file():continue
    try:
        candidate=pd.read_parquet(path) if path.suffix=='.parquet' else pd.read_csv(path,keep_default_na=False)
        assert len(candidate)==1005700 and candidate.path.is_unique
        assert candidate.label.isin([0,1]).all() and candidate.ok.isin([0,1]).all()
        assert signature(candidate)==EXPECTED_SIGNATURE
        df=candidate;manifest_used=path;break
    except Exception as exc:print(path.name,type(exc).__name__,str(exc)[:160],flush=True)
assert df is not None,'No valid matching manifest. Restore it; do not regenerate the split.'
assert ORIGINAL_CSV.is_file() and file_hash(ORIGINAL_CSV)==audit['source_splits_sha256']
fields=['url','ok','label','domain','fold','split']
raw=pd.read_csv(ORIGINAL_CSV,keep_default_na=False,usecols=['path']+fields)
assert raw.path.is_unique and len(raw)==audit['original_samples']
keys=df.source_path if 'source_path' in df else df.path
assert keys.is_unique
aligned=raw.set_index('path').loc[keys]
for col in fields:
    if col in ['ok','label','fold']:
        assert np.array_equal(pd.to_numeric(df[col]).to_numpy(),pd.to_numeric(aligned[col]).to_numpy()),col
    else:
        assert np.array_equal(df[col].fillna('').astype(str).to_numpy(),aligned[col].fillna('').astype(str).to_numpy()),col
assert df.groupby('sha256').split.nunique().max()==1
assert df.groupby('sha256').label.nunique().max()==1
assert df.groupby('domain',dropna=False).split.nunique().max()==1
assert df[df.ok.eq(1)].groupby('url').split.nunique().max()==1
test=df[df.split.eq('test')].reset_index(drop=True)
assert test.label.value_counts().to_dict()=={1:86120,0:60305}
assert len(test)==146425
sample=pd.concat([test[test.label.eq(label)].sample(n=SAMPLE_PER_CLASS,random_state=SAMPLE_SEED)
                  for label in [0,1]]).sort_values('path').reset_index(drop=True)
OUT=RUN/f'final_test_phase2_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:6]}'
OUT.mkdir(parents=True,exist_ok=False)
sample[['path','label','split','sha256']].to_csv(OUT/'test_sample_membership.csv',index=False)
write_json(dict(weights_phase1=str(OLD_WEIGHTS),weights_phase1_sha256=OLD_SHA,
    weights_phase2=str(NEW_WEIGHTS),weights_phase2_sha256=NEW_SHA,data_signature=EXPECTED_SIGNATURE,
    manifest=str(manifest_used),threshold=THRESHOLD,precision='FP32',batch_size=BATCH_SIZE,
    sample_per_class=SAMPLE_PER_CLASS,sample_seed=SAMPLE_SEED,variants=VARIANTS,
    translation_rule='Evaluate only samples with >=4 all-white border pixels on all sides; same subset for both models. Report excluded count.',
    shrink_rule='Resize 224->208 with NEAREST or BILINEAR; center on white 224 canvas.',
    history_note='Test was previously used for the phase-1 report. This is a documented re-evaluation after Validation-based augmentation design; no threshold or model search here.',
    versions=dict(torch=str(torch.__version__),torchvision=str(torchvision.__version__),opencv=cv2.__version__)),OUT/'evaluation_contract.json')
del raw,aligned,df,candidate;gc.collect()
print('PASS: 146425 Test images; fixed balanced sample of 1000. Output:',OUT,flush=True)

class MobileNetVisual(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual_backbone = torchvision.models.mobilenet_v3_small(weights=None)
        self.visual_backbone.classifier = nn.Identity()
        self.visual_proj = nn.Sequential(nn.Linear(576,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3))
        self.fusion = nn.Sequential(nn.Linear(256,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128,2))
    def forward(self, x):
        return self.fusion(self.visual_proj(self.visual_backbone(x)))


RESIZE=T.Resize((224,224))
NORMALIZE=T.Compose([T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])

class TestDataset(Dataset):
    def __init__(self,frame):self.paths=frame.path.to_numpy();self.labels=frame.label.to_numpy(dtype=np.int64)
    def __len__(self):return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im:x=NORMALIZE(RESIZE(im.convert('RGB')))
        return x,int(self.labels[i])

model=MobileNetVisual().to(DEVICE).eval()
for state in STATES.values():model.load_state_dict(state,strict=True)
loader=DataLoader(TestDataset(test),batch_size=BATCH_SIZE,shuffle=False,num_workers=WORKERS,
                  pin_memory=True,prefetch_factor=2 if WORKERS else None)

def metrics(y,p):
    pred=(p>=THRESHOLD).astype(np.int64)
    assert len(y)>0 and np.isfinite(p).all()
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(samples=int(len(y)),accuracy=float(accuracy_score(y,pred)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,average='macro',zero_division=0)),
        malicious_precision=float(precision_score(y,pred,zero_division=0)),
        malicious_recall=float(recall_score(y,pred,zero_division=0)),
        malicious_f1=float(f1_score(y,pred,zero_division=0)),roc_auc=float(roc_auc_score(y,p)),
        average_precision=float(average_precision_score(y,p)),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

def infer_loader(label):
    all_y=[];all_p=[];model.eval()
    with torch.inference_mode():
        for x,y in tqdm(loader,desc=label,mininterval=2):
            with torch.autocast(device_type='cuda',enabled=False):
                p=model(x.to(DEVICE,non_blocking=True)).softmax(1)[:,1]
            assert torch.isfinite(p).all()
            all_p.append(p.cpu().numpy());all_y.append(y.numpy())
    y=np.concatenate(all_y);p=np.concatenate(all_p)
    assert np.array_equal(y,test.label.to_numpy())
    return y,p

print('[3/5] Full original Test — both models...',flush=True)
full_rows=[];full_probabilities={}
for name in ['phase1','phase2_epoch2']:
    model.load_state_dict(STATES[name],strict=True)
    y,p=infer_loader(name+' FULL TEST')
    full_probabilities[name]=p
    m=metrics(y,p);full_rows.append(dict(model=name,**m))
    write_json(m,OUT/f'{name}_full_test_metrics.json')
    predictions=test[['path','label']].copy();predictions['malicious_probability']=p
    predictions['prediction']=(p>=THRESHOLD).astype(np.int64)
    predictions.to_csv(OUT/f'{name}_full_test_predictions.csv.gz',index=False,compression='gzip')
    report=classification_report(y,p>=THRESHOLD,labels=[0,1],target_names=['Benign','Malicious'],digits=5,zero_division=0)
    (OUT/f'{name}_classification_report.txt').write_text(report)
    fig,ax=plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(y,p>=THRESHOLD,labels=[0,1],display_labels=['Benign','Malicious'],
                                           cmap='Blues',values_format='d',colorbar=False,ax=ax)
    ax.set_title(name+' — Test');fig.tight_layout();fig.savefig(OUT/f'{name}_confusion_matrix.png',dpi=180);plt.close(fig)
    print(f"{name} FULL TEST: Acc={m['accuracy']:.4%} | Macro-F1={m['macro_f1']:.4%} | Malicious-F1={m['malicious_f1']:.4%} | Recall={m['malicious_recall']:.4%} | AUC={m['roc_auc']:.5f}",flush=True)
full_table=pd.DataFrame(full_rows);full_table.to_csv(OUT/'full_test_comparison.csv',index=False)

# Same Test sample for both models; no performance-dependent filtering.
print('[4/5] Prepare fixed sample, geometric eligibility and QR decoding...',flush=True)
base_images=[];translation_ok=[]
for row in tqdm(sample.itertuples(index=False),total=len(sample),desc='Sample images',mininterval=2):
    with Image.open(row.path) as im:image=RESIZE(im.convert('RGB'))
    base_images.append(image)
    bbox=ImageChops.difference(image,Image.new('RGB',(224,224),'white')).getbbox()
    translation_ok.append(bbox is None or (bbox[0]>=4 and bbox[1]>=4 and bbox[2]<=220 and bbox[3]<=220))
translation_ok=np.asarray(translation_ok,dtype=bool)
print('Translation eligibility:',int(translation_ok.sum()),'/',len(sample),flush=True)
assert len(np.unique(sample.label.to_numpy()[translation_ok]))==2,'Translation subset lacks both classes; inspect sample geometry.'
decoder=cv2.QRCodeDetector()

def decode(image):
    try:return decoder.detectAndDecode(cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2GRAY))[0]
    except cv2.error:return ''

def transform_image(image,variant):
    if variant=='original':return image.copy()
    shifts={'translate_left_4':(-4,0),'translate_right_4':(4,0),'translate_up_4':(0,-4),'translate_down_4':(0,4)}
    output=Image.new('RGB',(224,224),'white')
    if variant in shifts:
        output.paste(image,shifts[variant])
        assert np.any(np.asarray(image)!=255,axis=2).sum()==np.any(np.asarray(output)!=255,axis=2).sum()
    else:
        assert variant in ['shrink_nearest_208','shrink_bilinear_208']
        method=Image.Resampling.NEAREST if variant=='shrink_nearest_208' else Image.Resampling.BILINEAR
        output.paste(image.resize((208,208),resample=method),(8,8))
    return output

# Reuse full-Test predictions for the original sample to avoid unnecessary inference.
lookup=pd.Index(test.path).get_indexer(sample.path)
assert (lookup>=0).all()
base_probs={name:values[lookup] for name,values in full_probabilities.items()}
base_payloads=[decode(image) for image in tqdm(base_images,desc='Decode originals',mininterval=2)]
robust_rows=[];sample_predictions=sample[['path','label']].copy()
sample_predictions['translation_eligible']=translation_ok
for name in STATES:sample_predictions[name+'_original_probability']=base_probs[name]

print('[5/5] Paired robustness evaluation...',flush=True)
for variant in VARIANTS:
    mask=translation_ok if variant.startswith('translate_') else np.ones(len(sample),dtype=bool)
    indices=np.flatnonzero(mask);labels=sample.label.to_numpy(dtype=np.int64)[indices]
    if variant=='original':
        transformed=[base_images[i] for i in indices];payloads=[base_payloads[i] for i in indices]
    else:
        transformed=[transform_image(base_images[i],variant) for i in indices]
        payloads=[decode(image) for image in tqdm(transformed,desc='Decode '+variant,mininterval=2)]
    original_payloads=[base_payloads[i] for i in indices]
    readable=np.asarray([bool(p) for p in original_payloads])
    same_payload=np.asarray([bool(a) and a==b for a,b in zip(original_payloads,payloads)])
    details=pd.DataFrame(dict(path=sample.path.iloc[indices].to_numpy(),label=labels,
                             baseline_decoded=readable,same_payload=same_payload))
    for name in ['phase1','phase2_epoch2']:
        if variant=='original':p=base_probs[name][indices]
        else:
            model.load_state_dict(STATES[name],strict=True);model.eval();chunks=[]
            with torch.inference_mode():
                for start in tqdm(range(0,len(indices),BATCH_SIZE),desc=name+' '+variant,mininterval=2):
                    x=torch.stack([NORMALIZE(im) for im in transformed[start:start+BATCH_SIZE]]).to(DEVICE)
                    with torch.autocast(device_type='cuda',enabled=False):probs=model(x).softmax(1)[:,1]
                    chunks.append(probs.cpu().numpy())
            p=np.concatenate(chunks)
        m=metrics(labels,p);pred=p>=THRESHOLD;base_pred=base_probs[name][indices]>=THRESHOLD
        m.update(model=name,condition=variant,
            matched_original_accuracy=float(accuracy_score(labels,base_pred)),
            prediction_flip_rate=float(np.mean(pred!=base_pred)),
            baseline_decodable_samples=int(readable.sum()),same_payload_samples=int(same_payload.sum()),
            same_payload_fraction=float(same_payload.sum()/readable.sum()) if readable.any() else None,
            decoder_success_rate=float(np.mean([bool(v) for v in payloads])),
            flip_rate_same_payload=float(np.mean(pred[same_payload]!=base_pred[same_payload])) if same_payload.any() else None)
        robust_rows.append(m);details[name+'_probability']=p;details[name+'_prediction']=pred.astype(int)
        print(f"{name} {variant}: n={len(labels)} Acc={m['accuracy']:.4%} | Macro-F1={m['macro_f1']:.4%}",flush=True)
    details.to_csv(OUT/f'sample_{variant}_predictions.csv.gz',index=False,compression='gzip')
    pd.DataFrame(robust_rows).to_csv(OUT/'test_robustness_summary.csv',index=False)
    del transformed

robust_table=pd.DataFrame(robust_rows)
paired=robust_table.pivot(index='condition',columns='model',values='macro_f1').reindex(VARIANTS)
paired['delta_macro_f1_pp']=100*(paired['phase2_epoch2']-paired['phase1'])
paired.to_csv(OUT/'test_robustness_comparison.csv')
mean_rows=[]
for name in STATES:
    mean_rows.append(dict(model=name,mean_six_transformed_macro_f1=float(robust_table[
        robust_table.model.eq(name)&robust_table.condition.ne('original')].macro_f1.mean())))
pd.DataFrame(mean_rows).to_csv(OUT/'test_robustness_mean.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,p in full_probabilities.items():
    y=test.label.to_numpy();fpr,tpr,_=roc_curve(y,p);precision,recall,_=precision_recall_curve(y,p)
    axes[0].plot(fpr,tpr,label=name);axes[1].plot(recall,precision,label=name)
axes[0].plot([0,1],[0,1],'--',color='gray')
axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='Full Test ROC')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Full Test precision–recall')
for ax in axes:ax.legend()
fig.tight_layout();fig.savefig(OUT/'full_test_roc_pr.png',dpi=180);plt.close(fig)
fig,ax=plt.subplots(figsize=(12,5));positions=np.arange(len(VARIANTS))
ax.bar(positions-.2,100*paired.phase1,width=.4,label='phase1')
ax.bar(positions+.2,100*paired.phase2_epoch2,width=.4,label='phase2 epoch2')
ax.set_xticks(positions);ax.set_xticklabels(VARIANTS,rotation=25,ha='right')
ax.set(ylabel='Macro-F1 (%)',title='Paired fixed Test sample — reported sample sizes may differ by condition')
ax.legend();fig.tight_layout();fig.savefig(OUT/'test_robustness_comparison.png',dpi=180);plt.close(fig)

# Verify inputs unchanged and bundle outputs; weights are already backed up separately.
assert file_hash(NEW_WEIGHTS)==NEW_SHA and file_hash(OLD_WEIGHTS)==OLD_SHA
output_hashes={p.name:file_hash(p) for p in OUT.iterdir() if p.is_file()}
write_json(dict(status='completed',files_sha256=output_hashes,
               note='Descriptive paired comparisons; no significance test or model selection on these Test results.'),OUT/'completion_record.json')
zip_path=OUT.with_suffix('.zip');temporary=zip_path.with_suffix('.zip.writing')
with zipfile.ZipFile(temporary,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for path in OUT.iterdir():
        if path.is_file():archive.write(path,arcname=OUT.name+'/'+path.name)
with zipfile.ZipFile(temporary) as archive:assert archive.testzip() is None
os.replace(temporary,zip_path)
print('\n========== FINAL FULL TEST COMPARISON ==========')
shown=full_table[['model','accuracy','macro_f1','malicious_precision','malicious_recall','malicious_f1','roc_auc']].copy()
for col in ['accuracy','macro_f1','malicious_precision','malicious_recall','malicious_f1']:shown[col]*=100
print(shown.round(5).to_string(index=False))
print('\n========== TEST ROBUSTNESS (Macro-F1 %) ==========')
shown=paired.copy();shown[['phase1','phase2_epoch2']]*=100
print(shown.round(4).to_string())
print('Mean six transformed Macro-F1:',mean_rows)
print('Results folder:',OUT,flush=True)
print('Results ZIP:',zip_path,flush=True)
print('No training, threshold tuning, or checkpoint changes were performed.',flush=True)
